## Understanding RAG Workflow with StuffDocumentsChain

**RAG (Retrieval-Augmented Generation)** is a powerful workflow that combines:
-  **Retriever** (e.g., FAISS, ChromaDB): Fetches relevant documents from a large knowledge base.
-  **LLM** (e.g., Cohere, OpenAI): Answers questions using those documents.
-  **Document Chains** like `StuffDocumentsChain`: Combine documents into a prompt.

###  Components:
- **Document Loader**: Loads raw documents from files (e.g., `.txt`, `.pdf`, etc.).
- **Text Splitter**: Breaks long documents into chunks to fit within LLM context window.
- **Vector Store (FAISS)**: Stores vector embeddings of text chunks and retrieves the most similar ones.
- **Cohere Embeddings**: Convert text into vectors using Cohere's model (`embed-english-v3.0`).
- **Chat Model (Cohere)**: Generates answers based on provided documents and questions.
- **StuffDocumentsChain**: Concatenates all retrieved docs and "stuffs" them into a single prompt.
- **LangChain Expression Language (LCEL)**: Defines modular, reusable pipelines using operators like `|`.

This notebook demonstrates a complete RAG system using a finance dataset and `StuffDocumentsChain`.


In [ ]:
# !pip install -qU faiss-cpu

# Initialize the Cohere language model

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

from langchain_community.embeddings import OCIGenAIEmbeddings
embed_model = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',)


from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI


llm = ChatOCIGenAI(
      model_id='cohere.command-r-08-2024',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)    

In [ ]:
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import RetrievalQA
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
import os



# Load documents from a text file

In [ ]:

loader = TextLoader("finance_reports.txt")  # Use the correct local path
docs = loader.load()

# Split into chunks 

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(docs)

# Create FAISS vector store from the embedded chunks

In [ ]:
vectorstore = FAISS.from_documents(chunks, embed_model)
retriever = vectorstore.as_retriever()

# Define  prompt

In [ ]:

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant for finance analysis."),
    ("human", "Use the following financial data:\n{context}\n\nNow answer:\n{question}")
])


## Create a chain that combines all retrieved documents into one prompt feeds to context

In [ ]:

stuff_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

In [ ]:
from langchain_core.runnables import RunnableSequence
from langchain_core.runnables import RunnableLambda, RunnableMap

# Wrap it into a RAG pipeline


In [ ]:

#-------------Option1 without returning the source document------------
# rag_chain: RunnableSequence = (
#     {
#         "context": RunnableLambda(lambda x: retriever.get_relevant_documents(x["question"])),
#         "question": lambda x: x["question"]
#     }
#     | stuff_chain
# )

#-------------Option2 returning the source document------------
# Define the RAG pipeline logic: retrieve → generate → return
def rag_logic(input_dict):
    question = input_dict["question"]
    documents = retriever.get_relevant_documents(question)
    answer = stuff_chain.invoke({"context": documents, "question": question})
    return {
        "answer": answer,
        "sources": documents
    }

# Define the RAG pipeline logic: retrieve → generate → return

rag_chain = RunnableLambda(rag_logic)

# Run the full RAG pipeline with a sample question

In [ ]:

query = "Which companies had the highest revenue in 2023?"

response = rag_chain.invoke({"question": query})

# Print the final answer generated by the LLM

In [ ]:

print("Answer:\n", response["answer"])

# Print the source documents used in the response

print("\n📂 Source Documents:")
for i, doc in enumerate(response["sources"], 1):
    print(f"\n--- ✅Document {i} ---\n{doc.page_content[:500]}...\n")